# 04 Tree Boosted Models

In [1]:
# Notebook 04: Random Forest and boosted models

import joblib
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

from utils.data_utils import project_root
from utils.model_utils import regression_metrics, summarize_results

ROOT = project_root()
IN_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
TABLE_DIR = ROOT / "reports" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Loading PCA-transformed training and validation data from previous notebook
X_train = pd.read_parquet(IN_DIR / "X_train_pca.parquet")
X_val = pd.read_parquet(IN_DIR / "X_val_pca.parquet")
y_train = pd.read_parquet(IN_DIR / "y_train.parquet")["log_ic50"]
y_val = pd.read_parquet(IN_DIR / "y_val.parquet")["log_ic50"]

rows = []

# RF with hyperparameter tuning using CV
rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {
        "n_estimators": [200, 400],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
    },
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print(f"Random Forest best CV RMSE: {-rf.best_score_:.4f}")
rf_pred = rf.best_estimator_.predict(X_val)
rows.append({"model": "random_forest", "best_params": str(rf.best_params_), **regression_metrics(y_val, rf_pred)})
joblib.dump(rf.best_estimator_, MODELS_DIR / "rf_best.joblib")

#xgb wit hyperparameter tuning
xgb = GridSearchCV(
    XGBRegressor(random_state=42, eval_metric="rmse"),
    {
        "n_estimators": [100, 200],
        "learning_rate": [0.03, 0.1],
        "max_depth": [3, 5],
    },
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
xgb.fit(X_train, y_train)
print(f"XGBoost best CV RMSE: {-xgb.best_score_:.4f}")
xgb_pred = xgb.best_estimator_.predict(X_val)
rows.append({"model": "xgboost", "best_params": str(xgb.best_params_), **regression_metrics(y_val, xgb_pred)})
joblib.dump(xgb.best_estimator_, MODELS_DIR / "xgb_best.joblib")

results = summarize_results(rows)
results.to_csv(TABLE_DIR / "tree_boosted_model_results.csv", index=False)
results


Random Forest best CV RMSE: 1.2461
XGBoost best CV RMSE: 1.2661


,model,best_params,rmse,mae,r2
0,random_forest,"{'max_depth': None, 'min_samples_split': 2, 'n...",1.464334,1.101689,0.074343
1,xgboost,"{'learning_rate': 0.03, 'max_depth': 5, 'n_est...",1.467687,1.099861,0.070100


## Tree-Based Models — Results

### Models Trained
Two ensemble tree-based models were trained: Random Forest and XGBoost, both on 
the same 100 PCA components used in Notebook 03.

- **Random Forest** builds many independent decision trees on random subsets of 
  the data and averages their predictions. This reduces variance and helps 
  generalization compared to a single decision tree.
- **XGBoost** (Extreme Gradient Boosting) builds trees sequentially, where each 
  tree corrects the errors of the previous one. It is generally more accurate than 
  Random Forest but also more prone to overfitting without careful tuning.

Both models are non-parametric and do not assume a linear relationship between 
features and the target, unlike Ridge or Lasso.

### Hyperparameter Tuning
Hyperparameters were selected using GridSearchCV with 5-fold cross-validation, 
scoring on negative RMSE. The following grids were searched:

- **Random Forest**: n_estimators ∈ [200, 400] × max_depth ∈ [None, 10, 20] × 
  min_samples_split ∈ [2, 5]
- **XGBoost**: n_estimators ∈ [100, 200] × learning_rate ∈ [0.03, 0.1] × 
  max_depth ∈ [3, 5]

### Cross-Validation Results
| Model | Best CV RMSE | Best Hyperparameters |
|---|---|---|
| Random Forest | 1.2461 | max_depth=None, min_samples_split=2, n_estimators=200 |
| XGBoost | 1.2661 | learning_rate=0.03, max_depth=5, n_estimators=100 |

### Validation Set Results
| Model | RMSE | MAE | R² |
|---|---|---|---|
| Random Forest | 1.464 | 1.102 | 0.074 |
| XGBoost | 1.468 | 1.100 | 0.070 |

Overall, performance is lower than the linear models from Notebook 03, 
with R² ≈ 0.07 for both models.

### Key Observations

- **Tree models underperformed all linear and kernel models**: Random Forest and 
  XGBoost achieved R²≈0.07, compared to SVR at R²=0.127 and Ridge at R²=0.106 
  on the validation set. This is an important finding rather than a failure, as it reflects the structure of the feature space.

- **Why tree models underperformed**: PCA transforms the original 18,900 genes into 
  100 linear combinations (principal components). Tree-based models are designed to 
  find splits and interactions in raw feature space. After PCA, much of the nonlinear structure that tree models typically exploit is no longer explicitly present, as features are compressed into a lower-dimensional linear representation. 
  Linear and kernel models are inherently better suited to PCA-transformed inputs.

- **Random Forest selected max_depth=None** (fully grown trees with no depth limit), 
  and the gap between CV and validation RMSE (1.25 vs 1.46) suggests mild overfitting. 
  This is expected, as with only 617 samples, fully grown trees can memorize training 
  patterns that do not generalize. A depth limit (e.g. max_depth=10 or 20) was also 
  searched but was not selected by cross-validation.

- **XGBoost selected a low learning rate (0.03)**: This means each tree contributes 
  a small correction, requiring more trees to converge. Combined with max_depth=5, 
  the model chose a conservative, regularized configuration — yet still underperformed 
  linear models, reinforcing that the feature space structure favors linear approaches.

- **Design choice — why PCA before tree models**: PCA was applied primarily to address 
  the p >> n problem (18,900 features, 617 samples) and to enable linear models. 
  For tree models specifically, PCA is not always beneficial — trees can handle 
  high-dimensional sparse data natively. This highlights that preprocessing choices can influence which model classes perform best. An alternative approach would be to train 
  tree models directly on variance-filtered genes with feature selection, which may 
  yield better performance. This remains a direction for future work.

### Saved Artifacts
- `rf_best.joblib`, `xgb_best.joblib`
- `tree_boosted_model_results.csv`

## References

- Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. 
  *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge 
  Discovery and Data Mining (KDD 2016)*. https://arxiv.org/abs/1603.02754

- Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). Why tree-based models 
  still outperform deep learning on tabular data. *Advances in Neural Information 
  Processing Systems (NeurIPS 2022)*. https://arxiv.org/abs/2207.08815

- Tang, Y.-C., & Gottlieb, A. (2021). Explainable drug sensitivity prediction through cancer pathway enrichment. Scientific Reports, 11, 3128. https://doi.org/10.1038/s41598-021-82612-7